In [10]:
import re
import numpy as np
import pandas as pd
from gurobipy import Model, GRB

# =========================
# 0) FILE PATHS
# =========================
LISTINGS_CSV = "/Users/adriandarmali/Documents/Builder/vancouver_listings_with_neighbourhood_rates_2025_with_centroids.csv"
TRANSPORT_XLSX = '/Users/adriandarmali/Documents/Builder/transport data.xlsx' 
TRANSPORT_SHEET = "Sheet1"

# =========================
# 1) USER INPUTS
# =========================
N_RESULTS = 10          # total listings to return
MAX_PER_AREA = 2        # max listings per neighbourhood/area
Cmax = 50.0             # hard crime cap (incidents per 1,000)

Bmax = 1400.0           # per-listing budget cap (set to None to disable)

w_time = 50.0           # $ per minute (willingness-to-trade)
delta_m = 500.0         # $ penalty per extra mode (soft, objective only)

# =========================
# 2) HELPERS
# =========================
def canon(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s.replace("—", "-").replace("–", "-")

def read_csv_with_fallback(path):
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            pass
    # last resort (keeps going, may drop rows with bad lines)
    return pd.read_csv(path, encoding="latin-1", on_bad_lines="skip")

# =========================
# 3) LOAD DATA
# =========================
df_list = read_csv_with_fallback(LISTINGS_CSV)
df_tr = pd.read_excel(TRANSPORT_XLSX, sheet_name=TRANSPORT_SHEET)

# Canonical neighbourhood keys for join
df_list["Neighbourhood_c"] = df_list["Neighbourhood"].map(canon)
df_tr["Neighbourhood_c"] = df_tr["Neighbourhood1"].map(canon)

# Rename transport columns
df_tr = df_tr.rename(columns={
    "Neighbourhood1": "Neighbourhood_transport",
    "Avg. CrimeRate per 1000 2025 est": "CrimeRate",
    "Duration to UBC (mins)": "Duration",
    "# modes": "Modes",
})

# Join transport (area-level) onto listings (listing-level)
df = df_list.merge(
    df_tr[["Neighbourhood_c", "CrimeRate", "Duration", "Modes"]],
    on="Neighbourhood_c",
    how="left"
)

# Keep only listings with required fields
needed = ["Property", "Price_CAD", "Neighbourhood_c", "CrimeRate", "Duration", "Modes"]
df = df.dropna(subset=needed).reset_index(drop=True)

# Optional: enforce user budget as a simple pre-filter (safe + faster).
# If you want it purely as a constraint, set Bmax=None and keep the constraint section below.
if Bmax is not None:
    df = df[df["Price_CAD"] <= Bmax].reset_index(drop=True)

# Basic sanity: ensure numeric
df["Price_CAD"] = pd.to_numeric(df["Price_CAD"], errors="coerce")
df["CrimeRate"] = pd.to_numeric(df["CrimeRate"], errors="coerce")
df["Duration"] = pd.to_numeric(df["Duration"], errors="coerce")
df["Modes"] = pd.to_numeric(df["Modes"], errors="coerce")
df = df.dropna(subset=["Price_CAD", "CrimeRate", "Duration", "Modes"]).reset_index(drop=True)

# =========================
# 4) BUILD LISTING-LEVEL MILP
# =========================
L = len(df)
if N_RESULTS > L:
    raise ValueError(f"Not enough listings after cleaning to select {N_RESULTS}. Only {L} available.")

p = df["Price_CAD"].to_numpy(float)
c = df["CrimeRate"].to_numpy(float)
t = df["Duration"].to_numpy(float)
modes = df["Modes"].to_numpy(float)

# Per-listing objective cost (raw metrics only)
cost = p + w_time * t + delta_m * (modes - 1)

mdl = Model("listing_top10_max2perarea")
x = mdl.addVars(L, vtype=GRB.BINARY, name="x")

# Objective: minimize total cost over selected listings
mdl.setObjective(sum(cost[i] * x[i] for i in range(L)), GRB.MINIMIZE)

# Select exactly N_RESULTS listings
mdl.addConstr(sum(x[i] for i in range(L)) == N_RESULTS, name="pick_exactly_10")

# Max 2 per area (Neighbourhood_c)
areas = df["Neighbourhood_c"].astype(str).unique().tolist()
area_to_idx = {a: [] for a in areas}
for i, a in enumerate(df["Neighbourhood_c"].astype(str)):
    area_to_idx[a].append(i)

for a, idxs in area_to_idx.items():
    mdl.addConstr(sum(x[i] for i in idxs) <= MAX_PER_AREA, name=f"max_per_area_{a}")

# Crime hard cap, conditional on selection (no hard filtering)
MC = max(0.0, float(np.max(c - Cmax)))
for i in range(L):
    mdl.addConstr(c[i] <= Cmax + MC * (1 - x[i]), name=f"crime_cap_{i}")

# Optional: enforce per-listing budget in-model (if you didn't prefilter)
# If you prefiltered above, this constraint is redundant.
if Bmax is not None:
    MB = max(0.0, float(np.max(p - Bmax)))
    for i in range(L):
        mdl.addConstr(p[i] <= Bmax + MB * (1 - x[i]), name=f"budget_cap_{i}")

mdl.optimize()

# =========================
# 5) OUTPUT
# =========================
chosen = [i for i in range(L) if x[i].X > 0.5]
res = df.loc[chosen, ["Property", "Neighbourhood", "Neighbourhood_c", "Price_CAD", "CrimeRate", "Duration", "Modes"]].copy()
res["ObjCost"] = cost[chosen]
res = res.sort_values("ObjCost", ascending=True).reset_index(drop=True)

print("\nSelected listings (10 total; max 2 per area):")
print(res.to_string(index=False))

print("\nArea counts:")
print(res.groupby("Neighbourhood_c").size().sort_values(ascending=False).to_string())


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.5.0 24F74)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 345 rows, 162 columns and 486 nonzeros
Model fingerprint: 0xb9ea4919
Variable types: 0 continuous, 162 integer (162 binary)
Coefficient statistics:
  Matrix range     [1e+00, 9e+01]
  Objective range  [1e+03, 5e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+00, 1e+03]
Found heuristic solution: objective 28897.000000
Presolve removed 334 rows and 120 columns
Presolve time: 0.00s
Presolved: 11 rows, 42 columns, 77 nonzeros
Found heuristic solution: objective 18769.000000
Variable types: 0 continuous, 42 integer (35 binary)

Root relaxation: cutoff, 3 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0     cutoff    0